### Import and run model

In [2]:
# Forward pass with an untrained model.
from cs336_basics.model import TransformerLM
import torch

# device=torch.device("cuda")
device = torch.device("mps")
model = TransformerLM(
  vocab_size=10000,
  context_length=256,
  num_layers=8,
  d_model=768,
  num_heads=16,
  d_ff=1344,
  rope_theta=10000.0,
  device=device,
  dtype=torch.float32,
)

input = torch.randint(0, 10000, (1, 256), device=device)
output = model(input)
print(output.shape)

torch.Size([1, 256, 10000])


In [5]:
# Generate text using a trained model.
from cs336_basics.generation import generate_text
from cs336_basics.tokenizer import BPETokenizer
from cs336_basics.model import TransformerLM
from cs336_basics.io import ROOT_PATH, load_checkpoint

tok = BPETokenizer.load(ROOT_PATH / "tokenizer/tinystories_train_10000.pt")

model, _, _ = load_checkpoint(
  ROOT_PATH / "model/TinyStories/volcanic-dream-4/checkpoint_39999.pt",
  model_class=TransformerLM,
)
text = generate_text(
  model=model,
  tokenizer=tok,
  input_text= "Costanza",
  max_new_tokens=100,
  temperature=1.0,
  top_p=0.0,
)
print(model._init_args)
text

{'vocab_size': 10000, 'context_length': 256, 'num_layers': 8, 'd_model': 768, 'num_heads': 16, 'd_ff': 1344, 'rope_theta': 10000.0, 'rms_norm_eps': 1e-05, 'device': 'cpu', 'dtype': torch.float32}


"Costanza's imagination.\nOne day, Lucy was busy playing in the kitchen and spotted a colorful knew sick grill. Her dad put it on the grill and called the travel patient to check on the grill. He came to visit, but before they could leave, they heard a strange noise. \nThe counter was young and knew that something was wrong. It had been living with another country, called the scooters costs each minute she wanted. So Lucy and her dad decided to take the button to theions"

### Basic Benchmark

In [7]:
# Models that we will benchmark.
from dataclasses import dataclass

@dataclass
class ModelConfig:
  vocab_size: int
  d_model: int
  d_ff: int
  num_layers: int
  num_heads: int

BATCH_SIZE = 4
MODELS = {
  "small":  ModelConfig(10000, 768, 3072, 12, 12),
  "medium": ModelConfig(10000, 1024, 4096, 24, 16),
  "large":  ModelConfig(10000, 1280, 5120, 36, 20),
  "xl":     ModelConfig(10000, 1600, 6400, 48, 25),
  "2.7B":   ModelConfig(10000, 2560, 10240, 32, 32),
}

In [13]:
from cs336_basics.model import TransformerLM
from timeit import default_timer as timer
import numpy as np
import torch

def benchmarking_script(
  model_config: ModelConfig,
  context_length: int,
  measure_also_backward: bool,
  synchronize: bool = True,
  warmup_steps: int = 5,
  measure_steps: int = 10,
  batch_size: int = 4,
) -> (int, int):
  assert torch.cuda.is_available(), "CUDA must be available for running this Benchmark"
  device = torch.device("cuda")
  print("Creating dummy data and model...")
  input = torch.randint(0, model_config.vocab_size, (batch_size, context_length), device=device)
  model = TransformerLM(
    vocab_size=model_config.vocab_size,
    context_length=context_length,
    num_layers=model_config.num_layers,
    d_model=model_config.d_model,
    num_heads=model_config.num_heads,
    d_ff=model_config.d_ff,
    rope_theta=10000.0,
    device=device,
    dtype=torch.float32,
  )
  
  print("Warming up...")
  for _ in range(warmup_steps):
    output = model(input)
    if measure_also_backward:
      output.sum().backward()

  print("Measuring...")
  measures_s = []
  for _ in range(measure_steps):
    # Clear gradients manually to measure 'clean' write speed (avoid readiing
    # and summing gradients from the previous step).
    for p in model.parameters():
      p.grad = None
        
    # Make sure all operations have finished before starting to measure.
    torch.cuda.synchronize()

    start = timer()
    output = model(input)
    if measure_also_backward:
      output.sum().backward()
    if synchronize:
      torch.cuda.synchronize()
    end = timer()

    measures_s.append((end - start))
  return np.mean(measures_s), np.std(measures_s)

In [ ]:
import modal

# Create an app will all needed packages.
app_image = (
    modal.Image.debian_slim()
    # Install cs336-basics before pip_install_from_pyproject tries (and fails)
    # to install it from PyPI.
    .apt_install("git")
    .run_commands(
        "git clone https://github.com/NiccoloSacchi/assignment1-basics /root/assignment1-basics",
        "pip install -e /root/assignment1-basics",
    )
    # This installs only the dependencies list, ignoring the rest, i.e. it does
    # not know that cs336-basics should be installed from a local path.
    .pip_install_from_pyproject("/Users/niccolosacchi/assignment2-systems/pyproject.toml")
)
app = modal.App("cs336-systems", image=app_image)

In [17]:
@app.function(gpu="T4")  # Any GPU is fine for faster scheduling.
def without_sync():
  return benchmarking_script(
    model_config = MODELS["medium"],
    context_length = 256,
    measure_also_backward = False,
    synchronize = False,
  )

with app.run():
    output = without_sync.remote()

output

(np.float64(0.05909464249999985), np.float64(0.002082240659920715))

In [18]:
@app.function(gpu="T4")  # Any GPU is fine for faster scheduling.
def with_sync():
  return benchmarking_script(
    model_config = MODELS["medium"],
    context_length = 256,
    measure_also_backward = False,
    synchronize = True,
  )

with app.run():
    output = with_sync.remote()

output

(np.float64(0.2626476134000001), np.float64(0.0005595802305376522))

In [19]:
@app.function(gpu="T4")  # Any GPU is fine for faster scheduling.
def with_sync_and_backward():
  return benchmarking_script(
    model_config = MODELS["medium"],
    context_length = 256,
    measure_also_backward = True,
    synchronize = True,
  )

with app.run():
    output = with_sync.remote()

output

(np.float64(0.26100828599999987), np.float64(0.0010159617270745966))

In [27]:
model_config = MODELS["medium"]
model = TransformerLM(
  vocab_size=model_config.vocab_size,
  context_length=256,
  num_layers=model_config.num_layers,
  d_model=model_config.d_model,
  num_heads=model_config.num_heads,
  d_ff=model_config.d_ff,
  rope_theta=10000.0,
  # device="",
  dtype=torch.float32,
)

def check_model_size_mb(model):
    # Sum up the size of every parameter tensor
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    # Sum up the size of every buffer (like RoPE frequencies or masks)
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    
    total_size_mb = (param_size + buffer_size) / 1024**2
    print(f"Model Size: {total_size_mb:.2f} MB")
    return total_size_mb
  
check_model_size_mb(model)

Model Size: 1615.82 MB


1615.81640625